# Trajectory Analysis across models for Suo et. al.

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad

sc.settings.verbose = 3

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import InputTrajectories
from sctram.generate.real import sc_suo_developmental_complete

2025-03-01 20:17:53.937 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-03-01 20:17:53.938 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

In [6]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")
adata_suo_complete = sc_suo_developmental_complete(dataset_dir=dataset_dir)

2025-03-01 20:17:59.953 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/suo_developmental_complete.h5ad') already exists. Skipping download.


In [7]:
input_trajectories_path = os.path.join(dataset_dir, f"adata_suo_input_haematopoeitic_lineage.pkl")
with open(input_trajectories_path, "rb") as _file:
    lit = pickle.load(_file)

In [9]:
def plot_obsms(lin, plot_obs, adata, label_on_data=False, n_cols=4, subplot_size=(5, 4), prevent_bottom_legend=False):
    # Collect all categories and their global counts for consistent coloring
    lin = lin.replace('_lineage', '').lower()
    global_counts = {}
    
    # First pass to get all categories and their global counts
    for use_rep in adata[lin]:
        adata_obj = adata[lin][use_rep]
        counts = adata_obj.obs[plot_obs].astype(str).value_counts().to_dict()
        for cat, count in counts.items():
            global_counts[cat] = global_counts.get(cat, 0) + count
        break  # as all the reps have the same anndata.obs
    
    # Sort categories by global abundance descending
    unique_lvl = sorted(global_counts.keys(), key=lambda x: -global_counts[x])
    
    # Create color mapping based on global abundance
    n_cats = len(unique_lvl)
    palette = sns.color_palette(cc.glasbey_category10, n_cats)
    lvl_to_color = {cat: palette[i] for i, cat in enumerate(unique_lvl)}
    
    # Plotting parameters
    n_cols = n_cols
    subplot_size = subplot_size
    legend_ncol = 4
    
    use_reps = list(adata[lin].keys())
    if not use_reps:
        raise ValueError(f"no obsm for {lin!r}")
        
    n_subplots = len(use_reps)
    n_rows = int(np.ceil(n_subplots / n_cols))
    n_datapoints = len(next(iter(adata[lin].values())))  # Get count from first adata_obj
    
    # Dynamic point size calculation
    base_size = 1e5
    s = base_size / n_datapoints
    
    # Figure setup
    fig_width = subplot_size[0] * n_cols
    fig_height = subplot_size[1] * n_rows + 2
    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig)
    
    # Plot each use_rep with local abundance ordering
    for idx, use_rep in enumerate(use_reps):
        ax = fig.add_subplot(gs[idx // n_cols, idx % n_cols])
        adata_obj = adata[lin][use_rep]
        umap_coords = adata_obj.obsm['X_umap']
        
        # Get local counts for layering order
        local_counts = adata_obj.obs[plot_obs].astype(str).value_counts().sort_values(ascending=False)
        sorted_cats = local_counts.index.tolist()
        
        # Plot each category in abundance order (large to small)
        for cat in sorted_cats:
            mask = adata_obj.obs[plot_obs].astype(str) == cat
            ax.scatter(
                umap_coords[mask, 0],
                umap_coords[mask, 1],
                s=s,
                c=[lvl_to_color[cat]],
                alpha=0.4,
                linewidth=s*0.3,
                # rasterized=True,
                edgecolor='none'
            )
        
        # Add labels on data if enabled
        if label_on_data:
            texts = []
            centroid_info = []  # Stores (centroid, color) for each label
            
            for cat in sorted_cats:
                mask = adata_obj.obs[plot_obs].astype(str) == cat
                coords = umap_coords[mask]
                if len(coords) == 0:
                    continue
                
                # Compute centroid with outlier removal
                initial_centroid = np.mean(coords, axis=0)
                distances = np.linalg.norm(coords - initial_centroid, axis=1)
                std = np.std(distances)
                if std == 0:
                    final_centroid = initial_centroid
                else:
                    threshold = 3.0 * std
                    filtered = distances <= threshold
                    filtered_coords = coords[filtered]
                    final_centroid = np.mean(filtered_coords, axis=0) if len(filtered_coords) > 0 else initial_centroid
                
                # Plot enlarged centroid point
                ax.scatter(
                    final_centroid[0], final_centroid[1],
                    s=30*s,  # 10x normal size
                    c=[lvl_to_color[cat]],
                    alpha=1.0,
                    edgecolor='white',
                    linewidth=s*1,
                    zorder=5
                )
                
                # Create squeezed text using LaTeX \scalebox
                txt = ax.text(
                    final_centroid[0], final_centroid[1],
                    cat,
                    fontsize=6,
                    weight='bold',  # Make text bold
                    ha='center',
                    va='center',
                    color="black", #lvl_to_color[cat],
                    zorder=6
                )
                # Add a white outline effect around the text
                txt.set_path_effects([
                    path_effects.Stroke(linewidth=0.5, foreground='white'),
                    path_effects.Normal()
                ])
                
                texts.append(txt)
                centroid_info.append((final_centroid, lvl_to_color[cat]))
            
            # Adjust text positions without automatic arrows
            if texts:
                adjust_text(
                    texts, 
                    ax=ax,
                    force_text=(0.3, 0.3),   # Increase these values for stronger text movement
                    force_points=(1.8, 1.8),
                    lim=500
                )
                
                # Add manual colored arrows
                for txt, (centroid, color) in zip(texts, centroid_info):
                    # Get the text bounding box in data coordinates
                    renderer = fig.canvas.get_renderer()
                    bbox = txt.get_window_extent(renderer=renderer)
                    bbox_data = ax.transData.inverted().transform(bbox)
                    
                    # Calculate the center of the text bounding box
                    text_center = [
                        (bbox_data[0, 0] + bbox_data[1, 0]) / 2,  # x-center
                        (bbox_data[0, 1] + bbox_data[1, 1]) / 2   # y-center
                    ]
                    
                    # Draw arrow from centroid to text center
                    ax.annotate(
                        '',
                        xy=centroid,  # Arrow points to text center
                        xytext=text_center,  # Arrow starts at centroid
                        arrowprops=dict(
                            arrowstyle='->',
                            color="black", #color,
                            lw=0.5,
                            alpha=1,
                            connectionstyle="arc3,rad=0.15"  # Adjust curvature as needed
                        ),
                        zorder=4
                    )
        
        # Subplot formatting
        ax.set_title(use_rep, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines[:].set_visible(False)

    if not label_on_data and not prevent_bottom_legend:
        # Create unified legend only if labels are not on data
        legend_handles = [
            plt.Line2D([0], [0], 
             marker='o', 
             color='w', 
             markerfacecolor=lvl_to_color[cat], 
             markersize=10,
             label=f"{cat} ({global_counts[cat]:,})")
            for cat in unique_lvl
        ]
        
        fig.legend(
            handles=legend_handles,
            loc='lower center',
            ncol=legend_ncol,
            bbox_to_anchor=(0.5, -0.05),
            frameon=False,
            fontsize=10,
            handletextpad=0.1
        )

    # Final layout adjustments
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    fig.suptitle(f"{lin} - {plot_obs}", y=0.98, fontsize=12)
    plt.show()

In [10]:
def plot_trajectory(
        graph,
        title,
        figsize=(12, 4),
        font_size=10,
        node_size=300,
        node_color='#8DA0CB',  # Modern muted blue
        cmap='viridis',
        color_nodes_by_position=False,
        edge_width=1,
        edge_color='gray',
        label_offset=25,
        title_y=1.02,
        layout_args="-Grankdir=LR -Gnodesep=1.0 -Granksep=2.5"
    ):
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111)

    # Calculate hierarchical layout
    pos = graphviz_layout(graph, prog="dot", args=layout_args)

    # Calculate node colors using positional gradient
    if color_nodes_by_position:
        try:
            xs = [pos[node][0] for node in graph.nodes]
            min_x, max_x = min(xs), max(xs)
            if max_x != min_x:
                color_vals = [(x - min_x)/(max_x - min_x) for x in xs]
            else:  # Fallback if single column
                color_vals = [0.5]*len(xs)
            node_color = plt.get_cmap(cmap)(color_vals)
        except:
            print("Warning: Positional coloring failed. Using base color.")

    # Calculate label positions
    label_pos = {n: (x, y + label_offset) for n, (x, y) in pos.items()}

    # Draw network elements
    nx.draw_networkx_nodes(
        graph, pos, ax=ax,
        node_size=node_size,
        node_color=node_color,
        edgecolors='black',
        linewidths=0.8,
        alpha=1.0
    )

    nx.draw_networkx_edges(
        graph, pos, ax=ax,
        arrowstyle='-|>',
        arrowsize=15,
        edge_color=edge_color,
        width=edge_width,
        alpha=1.0,
        connectionstyle='arc3'
        # connectionstyle='arc3,rad=0.2'
    )

    nx.draw_networkx_labels(
        graph, label_pos, ax=ax,
        font_size=font_size,
        font_weight='normal',
        font_family=plt.rcParams['font.family'],
        alpha=1.0
    )

    # Add styled title
    fig.suptitle(
        title,
        y=title_y,
        fontsize=font_size + 2,
        fontweight='bold',
        color='#333333'
    )

    # Configure final layout
    plt.subplots_adjust(
        left=0.03,
        right=0.97,
        top=0.88 if title else 0.97,
        bottom=0.03
    )
    ax.set_axis_off()
    plt.show()

In [11]:
n_neighbors = 50
lineages = ["Haematopoeitic_lineage"]

In [12]:
adata_lineage = dict()
for lineage in lineages:
    lineage_part = lineage.replace('_lineage', '').lower()
    adata_lineage[lineage_part] = dict()
    for use_rep in adata_suo_complete.obsm.keys():
        output_file = os.path.join(dataset_dir, f"adata_suo_{lineage_part}_umap_{use_rep}.h5ad")
        if not os.path.exists(output_file):
            print(f"! - Not exists: {lineage!r} with {use_rep!r}")
        else:
            print(f"- Loaded: {lineage!r} with {use_rep!r}")
            adata_lineage[lineage_part][use_rep] = ad.read_h5ad(output_file)

- Loaded: 'Haematopoeitic_lineage' with 'Unintegrated'
- Loaded: 'Haematopoeitic_lineage' with 'X_pca'
- Loaded: 'Haematopoeitic_lineage' with 'harmony'
- Loaded: 'Haematopoeitic_lineage' with 'invae'
- Loaded: 'Haematopoeitic_lineage' with 'scanvi'
- Loaded: 'Haematopoeitic_lineage' with 'scvi'
- Loaded: 'Haematopoeitic_lineage' with 'tardis_1'
- Loaded: 'Haematopoeitic_lineage' with 'tardis_2'


In [15]:
lineage = "Haematopoeitic_lineage"
stem_cells = 'Haematopoetic_progenitors'
lvl1_haematopoeitic = np.array(adata_suo_complete[adata_suo_complete.obs["LVL0"] == lineage].obs["LVL1"].unique())
groups = {f"stem_cells_and_{i.lower()}": [stem_cells, i] for ind, i in enumerate(lvl1_haematopoeitic) if i != stem_cells}

adata_groups = dict()
for g in groups.keys():
    lineage_part = lineage.replace('_lineage', '').lower()
    adata_groups[g] = dict()
    for use_rep in adata_suo_complete.obsm.keys():
        output_file = os.path.join(dataset_dir, f"adata_suo_{lineage_part}_umap_{use_rep}_{g}.h5ad")
        if not os.path.exists(output_file):
            print(f"! - Not exists: {lineage!r} with {use_rep!r} of group {g!r}")
        else:
            print(f"- Loaded: {lineage!r} with {use_rep!r} of group {g!r}")
            adata_groups[g][use_rep] = ad.read_h5ad(output_file)

- Loaded: 'Haematopoeitic_lineage' with 'Unintegrated' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'X_pca' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'harmony' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'invae' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'scanvi' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'scvi' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'tardis_1' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'tardis_2' of group 'stem_cells_and_myeloid_differentiated_cells'
- Loaded: 'Haematopoeitic_lineage' with 'Unintegrated' of group 'stem_cells_and_mem'
- Loaded: 'Haematopoeitic_lineage' with 'X_pca' of group 'stem_cells_and_mem

In [25]:
df = pd.DataFrame()
lineage = "Haematopoeitic_lineage"
for trajectory in sorted(lit.graph["trajectories"]):
    lineage_part = lineage.replace("_lineage", "").lower()
    for use_rep in adata_suo_complete.obsm.keys():
        if use_rep == "Unintegrated":
            continue
        output_file = os.path.join(dataset_dir, f"_metric_adata_suo_{lineage_part}_{use_rep}_{trajectory}.pickle")
        df_trajectory_obsm = pd.read_pickle(output_file)
        df_trajectory_obsm["representation"] = use_rep
        df_trajectory_obsm["trajectory"] = trajectory
        df = pd.concat([df, df_trajectory_obsm])
df.reset_index(drop=True, inplace=True)
df['score'] = pd.to_numeric(df['score'], errors='raise')
df.sort_values(by=["trajectory", "path", "metric", "representation"], inplace=True, ignore_index=True)
df = df[["trajectory", "path", "metric", "representation", "score"]]

In [32]:
df["trajectory"].unique()

array(['alternative_myeloid', 'b_cell_specialization', 'b_cells', 'cd4',
       'complete_stem_trajectory', 'dendritic', 'early_b_cells',
       'early_lymphoid', 'early_stem_trajectory', 'elp_branching',
       'erythroid', 'erythroid_megakaryocyte', 'gmp_branching',
       'granulocyte_macrophage', 'granulocyte_mast',
       'granulocyte_monocytes', 'haematopoeitic_lineage', 'icl_nk',
       'macrophage_specialization', 'megakaryocyte', 'neutrophil',
       'stem_cells_and_lymphoid_differentiated_cells',
       'stem_cells_and_mem',
       'stem_cells_and_myeloid_differentiated_cells', 't_cell_mid',
       't_cell_nkt', 'tissue_macrophages'], dtype=object)

In [26]:
1

1

## Data loading

In [31]:
df_trajectory = df[df["trajectory"] == "b_cells"]
df_trajectory = df_trajectory.pivot(index=['path', 'metric'], columns='representation', values='score')
df_trajectory

representation                                      X_pca      harmony  \
path       metric                                                        
adjacency  accuracy                              0.867769     0.801653   
           average_shortest_path_difference      3.519108     3.481756   
           clustering_coeff_diff                 0.123038     0.203349   
           f1_score                              0.692308     0.600000   
           frobenius                             3.752799     3.968247   
           gdv_similarity                        0.899926     0.810600   
           gin_gnn_similarity                    0.956072     0.945270   
           graph_edit_distance                   8.000000    12.000000   
           hamming_distance                     16.000000    24.000000   
           jaccard_similarity                    0.529412     0.428571   
           l1_norm                              20.123344    25.450836   
           laplacian_spectral_emd                0.015372     0.027926   
           mantel_correlation                    0.058396     0.195056   
           maximum_common_subgraph_distance      8.000000    12.000000   
           permutation_marginalized_ssim         0.539210     0.486602   
           persistence_diagram_distance          5.818456     1.895933   
           precision                             0.562500     0.450000   
           random_walk_kernel_distance           0.854726     0.925401   
           recall                                0.900000     0.900000   
           spectral_distance                     1.948751     2.163900   
           weisfeiler_lehman_distance            0.593185     0.810701   
embedding  branch_silhouette_score                    NaN     0.011091   
           directionality_preservation          -0.219679    -0.304184   
           embedding_distance_correlation        0.721571     0.641080   
           gearys_c_embedding                    0.568228     0.609445   
           graph_based_trustworthiness           0.932537     0.880163   
           morans_i_embedding                    0.460522     0.363984   
           neighborhood_preservation_score       0.351612     0.305069   
           normalized_mean_curvature             0.033321     0.043114   
           sammons_stress                        0.155161     0.172991   
           trajectory_cardinality_validation     0.563201     0.610687   
           wasserstein_distance_embedding        9.460959     8.836190   
pseudotime cdf_cramer_von_mises               6759.155616  8536.733011   
           cdf_kolmogorov_smirnov                0.757682     0.889833   
           concordance_index                     0.917067     0.798782   
           dtw_distance                          0.062450     0.074071   
           gearys_c_pseudotime                   0.210817     0.593367   
           kendall_correlation                   0.768770     0.550738   
           mae                                   0.268697     0.405367   
           morans_i_pseudotime                   0.734856     0.400879   
           mse                                   0.098798     0.189873   
           mutual_information_kde                0.201041    -0.057300   
           normalized_mutual_information         0.346757     0.166154   
           pearson_correlation                   0.672096     0.256520   
           r_squared                            -2.451438    -9.680343   
           r_squared_with_spline                 0.816168     0.455389   
           spearman_correlation                  0.910198     0.697645   
           wasserstein_distance_pseudotime       0.303382     0.566860   

representation                                      invae       scanvi  \
path       metric                                                        
adjacency  accuracy                              0.917355     0.867769   
           average_shortest_path_difference      3.519509     3.523224   
   

In [33]:
def min_max_normalize(row):
    min_val = row.min()
    max_val = row.max()
    if max_val == min_val:
        return pd.Series(1, index=row.index) 
    else:
        return (row - min_val) / (max_val - min_val)

df_trajectory = df_trajectory.apply(min_max_normalize, axis=1)
df_trajectory

representation                                   X_pca   harmony     invae  \
path       metric                                                            
adjacency  accuracy                           0.571429  0.000000  1.000000   
           average_shortest_path_difference   0.900737  0.000000  0.910416   
           clustering_coeff_diff              0.475622  1.000000  0.360414   
           f1_score                           0.602291  0.195745  1.000000   
           frobenius                          0.533571  1.000000  0.000000   
           gdv_similarity                     0.811918  0.000000  0.691400   
           gin_gnn_similarity                 0.199622  0.000000  0.876505   
           graph_edit_distance                0.428571  1.000000  0.000000   
           hamming_distance                   0.428571  1.000000  0.000000   
           jaccard_similarity                 0.560701  0.170213  1.000000   
           l1_norm                            0.317089  1.000000  0.065628   
           laplacian_spectral_emd             0.412728  1.000000  0.434184   
           mantel_correlation                 0.622592  1.000000  0.537267   
           maximum_common_subgraph_distance   0.428571  1.000000  0.000000   
           permutation_marginalized_ssim      0.745404  0.343460  1.000000   
           persistence_diagram_distance       0.240177  0.000000  0.290751   
           precision                          0.464286  0.000000  1.000000   
           random_walk_kernel_distance        0.869059  1.000000  0.441714   
           recall                             1.000000  1.000000  1.000000   
           spectral_distance                  0.776626  1.000000  0.392929   
           weisfeiler_lehman_distance         0.301033  1.000000  0.226103   
embedding  branch_silhouette_score                 NaN  0.000000       NaN   
           directionality_preservation        0.209067  0.077976  0.681106   
           embedding_distance_correlation     0.554392  0.276249  1.000000   
           gearys_c_embedding                 0.310606  0.456685  0.000000   
           graph_based_trustworthiness        0.593224  0.000000  0.628329   
           morans_i_embedding                 0.866706  0.539921  1.000000   
           neighborhood_preservation_score    1.000000  0.000000  0.610768   
           normalized_mean_curvature          0.000000  0.023241  1.000000   
           sammons_stress                     0.358587  0.512377  0.277366   
           trajectory_cardinality_validation  0.000000  0.467758  0.281929   
           wasserstein_distance_embedding     0.934321  0.000000  1.000000   
pseudotime cdf_cramer_von_mises               0.510008  1.000000  0.309559   
           cdf_kolmogorov_smirnov             0.372304  1.000000  0.153511   
           concordance_index                  0.769947  0.000000  0.737361   
           dtw_distance                       0.690785  1.000000  0.831165   
           gearys_c_pseudotime                0.162721  1.000000  0.370026   
           kendall_correlation                0.769947  0.000000  0.737361   
           mae                                0.572890  1.000000  0.245124   
           morans_i_pseudotime                0.796446  0.000000  0.567357   
           mse                                0.490123  1.000000  0.150832   
           mutual_information_kde             0.599521  0.000000  0.577025   
           normalized_mutual_information      0.740032  0.000000  0.472724   
           pearson_correlation                0.627601  0.000000  0.806705   
           r_squared                          0.701887  0.000000  0.913133   
           r_squared_with_spline              0.765092  0.000000  0.640321   
           spearman_correlation               0.865084  0.000000  0.834953   
           wasserstein_distance_pseudotime    0.443432  1.000000  0.167395   

representation                                  scanvi      scvi  tardis_1  \
path       metric    

In [36]:
def find_max_min_models(row):
    max_models = row[row == row.max()].index.tolist()
    min_models = row[row == row.min()].index.tolist()
    if sorted(min_models) == sorted(max_models):
        max_models, min_models= "", ""
    return pd.Series([max_models, min_models], index=['Max_Scoring_Models', 'Min_Scoring_Models'])

df_trajectory_minmax = df_trajectory.apply(find_max_min_models, axis=1)

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.float_format', '{:.6f}'.format):
    assert np.all(df_trajectory_minmax.index == df_trajectory.index)
    display(pd.concat([df_trajectory_minmax, df_trajectory], axis=1))

Max_Scoring_Models  \
path       metric                                                       
adjacency  accuracy                                           [invae]   
           average_shortest_path_difference                  [scanvi]   
           clustering_coeff_diff                            [harmony]   
           f1_score                                           [invae]   
           frobenius                                        [harmony]   
           gdv_similarity                                      [scvi]   
           gin_gnn_similarity                                [scanvi]   
           graph_edit_distance                              [harmony]   
           hamming_distance                                 [harmony]   
           jaccard_similarity                                 [invae]   
           l1_norm                                          [harmony]   
           laplacian_spectral_emd                           [harmony]   
           mantel_correlation                               [harmony]   
           maximum_common_subgraph_distance                 [harmony]   
           permutation_marginalized_ssim                      [invae]   
           persistence_diagram_distance                      [scanvi]   
           precision                                          [invae]   
           random_walk_kernel_distance                      [harmony]   
           recall                             [X_pca, harmony, invae]   
           spectral_distance                                [harmony]   
           weisfeiler_lehman_distance                       [harmony]   
embedding  branch_silhouette_score                             [scvi]   
           directionality_preservation                     [tardis_2]   
           embedding_distance_correlation                     [invae]   
           gearys_c_embedding                              [tardis_1]   
           graph_based_trustworthiness                       [scanvi]   
           morans_i_embedding                                 [invae]   
           neighborhood_preservation_score                    [X_pca]   
           normalized_mean_curvature                          [invae]   
           sammons_stress                                  [tardis_1]   
           trajectory_cardinality_validation                   [scvi]   
           wasserstein_distance_embedding                     [invae]   
pseudotime cdf_cramer_von_mises                             [harmony]   
           cdf_kolmogorov_smirnov                           [harmony]   
           concordance_index                                 [scanvi]   
           dtw_distance                                     [harmony]   
           gearys_c_pseudotime                              [harmony]   
           kendall_correlation                               [scanvi]   
           mae                                              [harmony]   
           morans_i_pseudotime                               [scanvi]   
           mse                                              [harmony]   
           mutual_information_kde                          [tardis_2]   
           normalized_mutual_information                     [scanvi]   
           pearson_correlation                             [tardis_1]   
           r_squared                                           [scvi]   
           r_squared_with_spline                             [scanvi]   
           spearman_correlation                              [scanvi]   
           wasserstein_distance_pseudotime                  [harmony]   

                                             Min_Scoring_Models    X_pca  \
path       metric                                                          
adjacency  accuracy                                   [harmony] 0.571429   
           average_shortest_path_difference           [harmony] 0.900737   
           clustering_coeff_diff                       [scanvi] 0.475622   
           f1_score 

In [ ]:
# TODO: create a dict to indicata for which metrics lower is better for which one higher is better.
# TODO: check scib benchmarking 
# TODO: reduce the scores for each trajectory based on ranks, get trajectory ranking score
# TODO: reduce the trajectory ranking score overall.